<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/agentic_ai_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [1]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1) Build the KB retriever

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using relevant evidence.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

# Reasoning:
# 1. FakeEmbeddings: Simulates vectorization without calling an external API.
# 2. FAISS: A high-performance vector database that stores document embeddings for similarity search.
# 3. Retriever: Interface to query the vector store and return the top 'k' most relevant documents.
embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

/tmp/ipykernel_480/276859391.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


KB ready with 5 docs


## 2) Open source external tool: Wikipedia search

In [3]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    # WikipediaAPIWrapper provides an easy interface to search and retrieve page summaries.
    try:
        results = wiki.run(query)
        if "No good Wikipedia search result" in results:
            return [], "No results found."
        # We return a simple structured format for the generator
        snippets = [{'title': query, 'summary': results}]
        return snippets, None
    except Exception as e:
        return [], str(e)

print(wiki_search('Python programming')[0][:1])

[]


## 3) Simple planner (rule-based)

In [4]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest', 'kb', 'style']

def plan(question: str):
    # Logic: If the question contains keywords found in our internal KB, route to 'kb'.
    # Otherwise, default to 'wiki' for general external knowledge.
    question_lower = question.lower()
    if any(kw in question_lower for kw in kb_keywords):
        return {'action': 'kb'}
    return {'action': 'wiki'}

print(plan('How to ground answers?'))
print(plan('Who created Python?'))

{'action': 'kb'}
{'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets."
    "If there is little evidence, say so and suggest a follow-up query."
    "Cite sources like [kb:doc1] or [wiki:Title]."
    "Question: {question}"
    "Context:{context}"
    "Wiki:{wiki}"
    "Answer:"
)

def get_tiny_generator(model_id: str = 'sshleifer/tiny-gpt2'):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline('text-generation', model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 50):
    gen = get_tiny_generator()
    out = gen(prompt_text, max_new_tokens=max_new_tokens, truncation=True)
    full = out[0]["generated_text"]
    completion = full[len(prompt_text):].strip()
    return completion

def answer_question(question: str, use_tiny_model: bool = False):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action']=='kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action']=='wiki':
        wiki_snips, wiki_err = wiki_search(question)

    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'

    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    prompt_val = messages[0].content

    if use_tiny_model:
        final_answer = summarize_with_tiny(prompt_val)
    else:
        # FakeListChatModel provides a predictable output for testing RAG flow without latency.
        stub = FakeListChatModel(responses=['Based on the context, grounding is key [kb:retrievers]. If no info found, I suggest searching Wikipedia.'])
        final_answer = stub.invoke(messages).content

    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

## 5) Quick check on sample questions

In [6]:
tests = [
    "What are the key principles of agentic behavior?",
    "When was the French Revolution?",
    "Tell me about something that is not in the docs."
]

for q in tests:
    # Using False for use_tiny_model by default for speed/stability as requested
    res = answer_question(q, use_tiny_model=False)
    print('='*30)
    print(f"Question: {q}")
    print(f"Planner Decision: {res['plan']['action']}")
    print(f"KB Sources: {res['kb_sources']}")
    print(f"Wiki Sources: {res['wiki_sources']}")
    print(f"Answer: {res['answer']}")

Question: What are the key principles of agentic behavior?
Planner Decision: kb
KB Sources: ['kb:wikipedia_tip', 'kb:agentic_concept', 'kb:retrievers']
Wiki Sources: []
Answer: Based on the context, grounding is key [kb:retrievers]. If no info found, I suggest searching Wikipedia.
Question: When was the French Revolution?
Planner Decision: wiki
KB Sources: []
Wiki Sources: []
Answer: Based on the context, grounding is key [kb:retrievers]. If no info found, I suggest searching Wikipedia.
Question: Tell me about something that is not in the docs.
Planner Decision: wiki
KB Sources: []
Wiki Sources: []
Answer: Based on the context, grounding is key [kb:retrievers]. If no info found, I suggest searching Wikipedia.
